# Overview

This notebook will evaluated the size reductions achieve through various PNG optimization tools.

Tools like optipng and pngquant use several compression techniques including pixel depth reduction [https://appdevforall.atlassian.net/browse/ADFA-623] where possible to reduce PNG file size. See: [https://github.com/kornelski/pngquant/blob/main/README.md] and [https://optipng.sourceforge.net/pngtech/optipng.html]

## Stats collected
For every processed file, we collect:
- Original file name ``orig_file_name``
- Original file format ``format``
- Original file size ``orig_size``
- Original width ``orig_width``
- Original height ``orig_height``
- Original pixel mode ``orig_pixel_mode``
- Original pixel depth ``orig_pixel_depth``
- Compressed file name ``compressed_file_name``
- Compressed file size ``compressed_size``
- Compressed width ``compressed_width``
- Compressed height ``compressed_height``
- Compressed pixel mode ``compressed_pixel_mode``
- Compressed pixel depth ``compressed_pixel_depth``
- Tool ran to process file ``tool_used``
- Full command ran to process file ``command``
- stdout from command``stdout``
- stderr from command ``stderr``
- Tool runtime ``runtime``

In [121]:
import matplotlib
import numpy as np
import os
import pandas as pd
import subprocess
import sys
import time

from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from matplotlib import rcParams

In [63]:
# Directory with a few test images for this notebook. Currently a random sample of 5 images from Kotlin docs.
TEST_DIR = "test"

# https://stackoverflow.com/questions/1996577/how-can-i-get-the-depth-of-a-jpg-file
MODE_TO_BPP = {"1": 1, "L": 8, "P": 8, "RGB": 24, "RGBA": 32, "CMYK": 32, "YCbCr": 24, "LAB": 24, "HSV": 24, "I": 32, "F": 32}

In [70]:
"""
load_images(in_dir, ext)

Get all images with specific extension in specified directory as a list() of file names. Not recursive.

Returns: a list() of image files in the given directory that match the given extension.
"""

def load_images(in_dir, ext):
    return [os.path.join(in_dir, f) for f in os.listdir(in_dir) if (os.path.isfile(os.path.join(in_dir, f)) and f.split(".")[1] == ext)]

In [71]:
load_images(TEST_DIR, "png")

['test/github-notebook.png',
 'test/gaussian-distribution-output.png',
 'test/wasm-step-into.png',
 'test/gradle-kts-load-config.png',
 'test/wasm-debugger-improved.png']

In [32]:
"""
get_image_info(image)

Get basic image information: file name, file size, width, height, pixel depth

Returns: Given image's file name, file size, width, height, pixel depth
"""
def get_image_info(filename):
    with Image.open(filename) as img:
        width, height = img.size
        pixel_mode = img.mode
        image_format = img.format
    size = os.path.getsize(filename)
    return size, width, height, pixel_mode, image_format

In [13]:
"""
run_command(command)

Runs a command and returns stdout, stderr, and runtime.

This will be called to run the various PNG optimization tools. 

Caller can handle the collection of all other information.

Returns: command stdout, stderr, and runtime
"""

def run_command(command):
    t_0 = time.perf_counter()
    process = subprocess.run(command, shell=True, capture_output=True, text=True)
    t_1 = time.perf_counter()
    runtime = t_1 - t_0

    return process.stdout, process.stderr, runtime

## Tool-specific processing functions

All ``run_*`` functions share a signature: give the input file name, output file name, and any extra parameters you want to pass to the tool as they would appear in the invocation of the tool.

In [104]:
"""
run_optipng(in_image, out_image, params)

Run optipng over a given image and store the result at specified location on disk

'in_image' and 'out_image' are the input/output filenames.

'params' is a string of command line options passed to optipng (except for -out which is hardcoded to be set to 'out_image'

Returns: dict() with all information described in top of this document. (Keys are given in the inline code segments.)

"""

def run_optipng(in_image, out_image, params):
    command = "optipng -out " + out_image + " " + params + " " + in_image
    stdout, stderr, runtime = run_command(command)
    
    return command, stdout, stderr, runtime

In [105]:
# pngquant images/wasm-debug-controls.png --output wasm.png

"""
run_pngquant(in_image, out_image, params)

Run pngquant over a given image and store the result at specified location on disk

'in_image' and 'out_image' are the input/output filenames.

'params' is a string of command line options passed to optipng (except for -out which is hardcoded to be set to 'out_image'

Returns: dict() with all information described in top of this document. (Keys are given in the inline code segments.)
"""

def run_pngquant(in_image, out_image, params):
    command = "pngquant " + in_image + " --output " + out_image + " " + params
    stdout, stderr, runtime = run_command(command)
    
    return command, stdout, stderr, runtime

## Construct main dataframe

In [46]:
columns = ["orig_file_name", "format", "orig_size", "orig_width", "orig_height", "orig_pixel_mode", "orig_pixel_depth", "out_image", 
    "compressed_size", "compressed_width", "compressed_height", "compressed_pixel_mode", "compressed_pixel_depth", "tool_name", "command", "stdout",
    "runtime"]
image_df = pd.DataFrame(columns=columns)

## Test runs

In [103]:
"""
process_images(images, out_dir, process_fn, params, tool_name)

Apply process_fn to a list of images.

Params:
- images: A list() of filenames for the images to process
- out_dir: output directory for processed images
- process_fn: function name for processor to apply (this function fills in a command template, runs the command, and records stdout/stderr/
runtime
- params: command line parameters passed to process_fn. process_fn will put these in the correct segment of the command template.
- tool_name: name of tool used, used to populate tool_name column

Returns:
- image_df: A data frame with all recorded information specified at the top.
"""
def process_images(images, out_dir, process_fn, params, tool_name):
    if not os.path.exists(out_dir):
        os.makedirs(out_dir)

    image_df = pd.DataFrame(columns=columns)
    
    for image_name in images:
        out_image = os.path.join(out_dir, os.path.basename(image_name))
        command, stdout, stderr, runtime = process_fn(image_name, out_image, params)
        
        orig_size, orig_width, orig_height, orig_pixel_mode, image_format = get_image_info(image_name)
        orig_pixel_depth = MODE_TO_BPP[orig_pixel_mode]
        
        if os.path.exists(out_image):
            compressed_size, compressed_width, compressed_height, compressed_pixel_mode, image_format = get_image_info(out_image) 
            compressed_pixel_depth = MODE_TO_BPP[compressed_pixel_mode]
        else:
            compressed_size, compressed_width, compressed_height, compressed_pixel_mode, image_format = ["", "", "", "", ""]
            compressed_pixel_depth = ""
    
        row_items = [image_name, image_format,  orig_size, orig_width, orig_height, orig_pixel_mode, orig_pixel_depth, out_image, 
        compressed_size, compressed_width, compressed_height, compressed_pixel_mode, compressed_pixel_depth, tool_name, command, stdout,
        runtime]
    
        row = dict(zip(columns, row_items))
        image_df = pd.concat([image_df, pd.DataFrame([row])], ignore_index=True)

    return image_df

In [84]:
images = load_images("/home/elissa/ADFA/ADFA-ODT/SourceDocs/KotlinDocs/html/images", "png")
out_dir = "kotlin_compressed"

image_df = process_images(images, out_dir, run_optipng, "", "optipng")

/tmp/ipykernel_92239/1366654638.py:26: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  image_df = pd.concat([image_df, pd.DataFrame([row])], ignore_index=True)


In [86]:
image_df.to_csv("kotlin_optipng_defaults.tsv", sep="\t", index=None)

In [92]:
image_df["orig_pixel_depth"].value_counts()

orig_pixel_depth
32    205
24      7
Name: count, dtype: int64

In [90]:
image_df["compressed_pixel_depth"].value_counts()

compressed_pixel_depth
24    148
32     44
8      19
        1
Name: count, dtype: int64

In [98]:
filtered_df = image_df[image_df["compressed_size"] != ""]

In [99]:
filtered_df["size_reduction"] = (filtered_df["orig_size"].astype(np.int32) - filtered_df["compressed_size"].astype(np.int32))/filtered_df["orig_size"].astype(np.int32)

/tmp/ipykernel_92239/1221884383.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df["size_reduction"] = (filtered_df["orig_size"].astype(np.int32) - filtered_df["compressed_size"].astype(np.int32))/filtered_df["orig_size"].astype(np.int32)


In [102]:
orig_sum = filtered_df["orig_size"].sum()
compressed_sum = filtered_df["compressed_size"].sum()
print("Total original size: " + str(orig_sum))
print("Total compressed size: " + str(compressed_sum))
print("Total reduction: " + str((float(orig_sum) - compressed_sum)/orig_sum))

Total original size: 38243244
Total compressed size: 24862795
Total reduction: 0.34987745809429766


In [107]:
images = load_images("/home/elissa/ADFA/ADFA-ODT/SourceDocs/KotlinDocs/html/images", "png")
out_dir = "kotlin_pngquant"

image_df = process_images(images, out_dir, run_pngquant, "", "pngquant")

/tmp/ipykernel_92239/1349110588.py:42: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  image_df = pd.concat([image_df, pd.DataFrame([row])], ignore_index=True)


In [108]:
filtered_df = image_df[image_df["compressed_size"] != ""]

In [109]:
orig_sum = filtered_df["orig_size"].sum()
compressed_sum = filtered_df["compressed_size"].sum()
print("Total original size: " + str(orig_sum))
print("Total compressed size: " + str(compressed_sum))
print("Total reduction: " + str((float(orig_sum) - compressed_sum)/orig_sum))

Total original size: 38243244
Total compressed size: 13203169
Total reduction: 0.6547581319199804


In [110]:
image_df["orig_pixel_depth"].value_counts()

orig_pixel_depth
32    205
24      7
Name: count, dtype: int64

In [111]:
image_df["compressed_pixel_depth"].value_counts()

compressed_pixel_depth
8    211
       1
Name: count, dtype: int64